In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F
import pandas as pd
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import torch
from torch.utils.data import TensorDataset, DataLoader


In [2]:
train=pd.read_csv("./data/preprocessedTrain.csv")
test=pd.read_csv("./data/preprocessedTest.csv")

# To fill nulls for CNN
print(train.isnull().sum())
train['f_Header_b_payload_Ratio'] = train['f_Header_b_payload_Ratio'].replace(np.nan, train['f_Header_b_payload_Ratio'].max() + 1) 
train['b_Header_f_payload_Ratio'] = train['b_Header_f_payload_Ratio'].replace(np.nan, train['b_Header_f_payload_Ratio'].max() + 1) 
print(train.isnull().sum())

test['f_Header_b_payload_Ratio'] = test['f_Header_b_payload_Ratio'].replace(np.nan, test['f_Header_b_payload_Ratio'].max() + 1) 
test['b_Header_f_payload_Ratio'] = test['b_Header_f_payload_Ratio'].replace(np.nan, test['b_Header_f_payload_Ratio'].max() + 1) 

id.orig_p                       0
id.resp_p                       0
service                         0
flow_duration                   0
fwd_pkts_tot                    0
                            ...  
f_Header_b_payload_Ratio    84484
b_Header_f_payload_Ratio     5446
bwd_payload_zero_flg            0
fwd_payload_zero_flg            0
attack_type                     0
Length: 83, dtype: int64
id.orig_p                   0
id.resp_p                   0
service                     0
flow_duration               0
fwd_pkts_tot                0
                           ..
f_Header_b_payload_Ratio    0
b_Header_f_payload_Ratio    0
bwd_payload_zero_flg        0
fwd_payload_zero_flg        0
attack_type                 0
Length: 83, dtype: int64


In [3]:
X_train = train.drop("attack_type", axis=1).values
y_train = train["attack_type"].values

X_test = test.drop("attack_type", axis=1).values
y_test = test["attack_type"].values

In [ ]:

train = pd.read_csv("./data/preprocessedTrain.csv")
test = pd.read_csv("./data/preprocessedTest.csv")

train = train.fillna(0)
test = test.fillna(0)

#Set up binary classification
def to_binary_label(y):
    # 0 = benign, 1 = malicious
    return [0 if val in [3,11,12] else 1 for val in y]

y_train = to_binary_label(train["attack_type"].values)
y_test = to_binary_label(test["attack_type"].values)

X_train = train.drop("attack_type", axis=1).values
X_test = test.drop("attack_type", axis=1).values


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#SMOTE
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

#Tensor conversion
X_train_tensor = torch.tensor(X_train_res, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_res, dtype=torch.float32).view(-1,1)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1,1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)


In [ ]:
#model
class BinaryAttackClassifier(nn.Module):
    def __init__(self, input_size=83, hidden_size=128):
        super(BinaryAttackClassifier, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, 1)  # raw logits

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)  # raw logits
        return x

model = BinaryAttackClassifier(input_size=X_train.shape[1])
criterion = nn.BCEWithLogitsLoss()  # handles logits internally
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)


In [ ]:
model = BinaryAttackClassifier(input_size=X_train_tensor.shape[1])
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Split mini-validation set from training
X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train_tensor, y_train_tensor, test_size=0.1, random_state=42
)
train_loader = DataLoader(TensorDataset(X_train_sub, y_train_sub), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=64)

#training
epochs = 20
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # Mini-validation
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            outputs = model(inputs)
            loss_val = criterion(outputs, labels)
            val_loss += loss_val.item()
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total
    print(f"Epoch [{epoch+1}/{epochs}], "
          f"Train Loss: {running_loss/len(train_loader):.4f}, "
          f"Val Loss: {val_loss/len(val_loader):.4f}, "
          f"Val Acc: {val_acc*100:.2f}%")


Epoch [1/20], Train Loss: 0.0117, Val Loss: 0.0059, Val Acc: 99.88%
Epoch [2/20], Train Loss: 0.0025, Val Loss: 0.0064, Val Acc: 99.93%
Epoch [3/20], Train Loss: 0.0025, Val Loss: 0.0011, Val Acc: 99.96%
Epoch [4/20], Train Loss: 0.0015, Val Loss: 0.0103, Val Acc: 99.92%
Epoch [5/20], Train Loss: 0.0016, Val Loss: 0.0076, Val Acc: 99.89%
Epoch [6/20], Train Loss: 0.0012, Val Loss: 0.0103, Val Acc: 99.97%
Epoch [7/20], Train Loss: 0.0016, Val Loss: 0.0255, Val Acc: 99.96%
Epoch [8/20], Train Loss: 0.0010, Val Loss: 0.0302, Val Acc: 99.97%
Epoch [9/20], Train Loss: 0.0011, Val Loss: 0.0236, Val Acc: 99.97%
Epoch [10/20], Train Loss: 0.0011, Val Loss: 0.0117, Val Acc: 99.97%
Epoch [11/20], Train Loss: 0.0011, Val Loss: 0.0192, Val Acc: 99.97%
Epoch [12/20], Train Loss: 0.0008, Val Loss: 0.0232, Val Acc: 99.98%
Epoch [13/20], Train Loss: 0.0010, Val Loss: 0.0138, Val Acc: 99.97%
Epoch [14/20], Train Loss: 0.0007, Val Loss: 0.0161, Val Acc: 99.98%
Epoch [15/20], Train Loss: 0.0011, Val Loss

In [7]:
from sklearn.metrics import classification_report

all_labels = []
all_preds = []

model.eval()
with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs)
        preds = (outputs > 0.5).float()
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

print(classification_report(all_labels, all_preds, target_names=['Benign', 'Attack']))


              precision    recall  f1-score   support

      Benign       0.99      0.99      0.99       880
      Attack       1.00      1.00      1.00     23744

    accuracy                           1.00     24624
   macro avg       1.00      1.00      1.00     24624
weighted avg       1.00      1.00      1.00     24624

